# Full Run

This notebook consists of the following stages:

1. load encoders
2. make and download splits
3. Training downstream classifiers with linear probing
4. Summarizing and plotting probe results
5. Explainability on evaluation (test) set

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.hub import download_url_to_file
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if not SRC_ROOT.exists():
    raise FileNotFoundError(f"Could not find src/ under {PROJECT_ROOT}")

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from cv.config import ARTIFACTS_ROOT, DEFAULT_ENCODER_CHECKPOINTS
from cv.data import build_downstream_datasets, create_fixed_split_indices, load_fixed_split_indices
from cv.encoders import load_encoder
from cv.explain import (
    discover_stage4_runs,
    generate_explanations_for_runs,
    run_explanation_qc,
    write_explanation_qc_report,
)
from cv.models import build_downstream_model, resolve_mode_config
from cv.train.metrics import summarize_test_accuracy_by_condition
from cv.train.trainer import (
    TrainingRunConfig,
    build_run_table_row,
    resolve_training_recipe,
    train_one_run,
)
from cv.utils.io import ensure_parent, read_json, write_json

## Runtime Config 

Load the following RUN_CONFIG for training, which is the same as flags for the CLI. 

In [2]:
RUN_CONFIG: dict[str, object] = {
    
    # encoders to test on 
    "conditions": ["supervised", "moco", "swav", "random_init"],
    "seeds": [0, 1, 2],

    "device": "cpu",
    "num_workers": 0, # number of subprocesses Pytorch Dataloader can use 
    "pin_memory": False, # set to false for CPU only run 

    # data download 
    "artifacts_root": None, # set to None for default, artifacts/
    "data_root": None, # default data/raw/
    "download_data": True, 

    # loading encoders; by default, downloaded to data/external 
    "prepare_checkpoints": True,
    "force_download_checkpoints": False, # controls whether existing local files 
    # are reused or replaced if prepare_checkpoints=True. Re-downloads encoders if True.
    "inspect_encoders": True,
    "allow_remote_download": True,

    # data splits 
    "create_splits": True,
    "overwrite_splits": False, # if split indice JSON exists, do not overwrite 
    "split_seed": 42,
    "val_ratio": 0.2,

    # Stage 4
    "run_cross_condition_sanity_check": True, # runs one batch no-grad forward pass per condition 
    # before a full training to make sure the trainng works; verifies logits shape is (batch_size, 10)
    "run_training": True,
    "sanity_checks": True,

    # linear probe training hyperparams override for each training 
    "probe_recipe_id": None,
    "random_init_recipe_id": None,
}

ARTIFACTS_ROOT_RESOLVED = Path(RUN_CONFIG["artifacts_root"]) if RUN_CONFIG["artifacts_root"] else ARTIFACTS_ROOT
print(json.dumps(RUN_CONFIG, indent=2))
print(f"Resolved artifacts root: {ARTIFACTS_ROOT_RESOLVED}")

{
  "conditions": [
    "supervised",
    "moco",
    "swav",
    "random_init"
  ],
  "seeds": [
    0,
    1,
    2
  ],
  "device": "cpu",
  "num_workers": 0,
  "pin_memory": false,
  "artifacts_root": null,
  "data_root": null,
  "download_data": true,
  "prepare_checkpoints": true,
  "force_download_checkpoints": false,
  "inspect_encoders": true,
  "allow_remote_download": true,
  "create_splits": true,
  "overwrite_splits": false,
  "split_seed": 42,
  "val_ratio": 0.2,
  "run_cross_condition_sanity_check": true,
  "run_training": true,
  "sanity_checks": true,
  "probe_recipe_id": null,
  "random_init_recipe_id": null
}
Resolved artifacts root: /Users/tlam/cv/artifacts


## 1. Encoder Preparation and Inspection

In [3]:
ENCODER_CONDITIONS = ["supervised", "moco", "swav"]

def _download_checkpoint(url: str, destination: Path, *, force: bool) -> None:
    if destination.exists() and not force:
        print(f"[skip] exists: {destination}")
        return

    ensure_parent(destination)
    print(f"[download] {url} -> {destination}")
    download_url_to_file(url, str(destination), progress=True)


def prepare_encoder_checkpoints(*, force_download: bool) -> None:
    cfg = DEFAULT_ENCODER_CHECKPOINTS
    _download_checkpoint(
        cfg.moco_checkpoint_url,
        cfg.moco_checkpoint_path,
        force=force_download,
    )
    _download_checkpoint(
        cfg.swav_checkpoint_url,
        cfg.swav_checkpoint_path,
        force=force_download,
    )


def inspect_encoders(
    *,
    conditions: list[str],
    device: str,
    allow_remote_download: bool,
    batch_size: int = 2,
) -> tuple[pd.DataFrame, dict[str, object]]:
    rows: list[dict[str, object]] = []
    failures: list[dict[str, str]] = []
    torch_device = torch.device(device)

    for condition in conditions:
        kwargs: dict[str, object] = {"freeze": True, "device": torch_device}
        if condition == "moco":
            kwargs["checkpoint_path"] = str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path)
            kwargs["allow_remote_download"] = allow_remote_download
        elif condition == "swav":
            kwargs["checkpoint_path"] = str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path)
            kwargs["allow_remote_download"] = allow_remote_download

        try:
            loaded = load_encoder(condition, **kwargs)
            images = torch.randn(batch_size, 3, 224, 224, device=torch_device)
            with torch.no_grad():
                features = loaded.encoder(images)

            if tuple(features.shape) != (batch_size, loaded.metadata.feature_dim):
                raise ValueError(
                    f"Unexpected feature shape for {condition}: {tuple(features.shape)}"
                )

            row = {
                "condition": condition,
                "feature_shape": list(features.shape),
                "feature_dim": loaded.metadata.feature_dim,
                "gradcam_layer": loaded.metadata.gradcam_target_layer,
                "encoder_frozen": not any(p.requires_grad for p in loaded.encoder.parameters()),
                "checkpoint_id": loaded.metadata.checkpoint_id,
                "checkpoint_origin": loaded.metadata.checkpoint_origin,
                "pretraining_objective": loaded.metadata.pretraining_objective,
            }
            rows.append(row)
        except Exception as exc:
            failures.append({"condition": condition, "error": str(exc)})

    report = {
        "stage": "stage-1-encoder-preparation",
        "device": device,
        "results": rows,
        "failures": failures,
    }
    return pd.DataFrame(rows), report


if RUN_CONFIG["prepare_checkpoints"]:
    prepare_encoder_checkpoints(force_download=bool(RUN_CONFIG["force_download_checkpoints"]))
else:
    print("Skipping checkpoint downloads (prepare_checkpoints=False).")

if RUN_CONFIG["inspect_encoders"]:
    encoder_df, encoder_report = inspect_encoders(
        conditions=ENCODER_CONDITIONS,
        device=str(RUN_CONFIG["device"]),
        allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
    )
    display(encoder_df)

    encoder_report_path = ARTIFACTS_ROOT_RESOLVED / "metrics" / "encoder_prep_report_notebook.json"
    write_json(encoder_report_path, encoder_report)
    print(f"Saved encoder report to: {encoder_report_path}")

    if encoder_report["failures"]:
        raise RuntimeError(f"Encoder inspection failures: {encoder_report['failures']}")
else:
    print("Skipping encoder inspection (inspect_encoders=False).")

[skip] exists: /Users/tlam/cv/data/external/moco_v2_800ep_pretrain.pth.tar
[skip] exists: /Users/tlam/cv/data/external/swav_800ep_pretrain.pth.tar


,condition,feature_shape,feature_dim,gradcam_layer,encoder_frozen,checkpoint_id,checkpoint_origin,pretraining_objective
0,supervised,"[2, 2048]",2048,encoder.layer4[-1].conv3,True,ResNet50_Weights.IMAGENET1K_V2,torchvision,supervised classification
1,moco,"[2, 2048]",2048,encoder.layer4[-1].conv3,True,/Users/tlam/cv/data/external/moco_v2_800ep_pre...,local file,MoCo v2 contrastive self-supervision
2,swav,"[2, 2048]",2048,encoder.layer4[-1].conv3,True,/Users/tlam/cv/data/external/swav_800ep_pretra...,local file,SwaV self-supervised clustering


Saved encoder report to: /Users/tlam/cv/artifacts/metrics/encoder_prep_report_notebook.json


Note: moco and swav were downloaded previously via CLI and were cached locally on device.

## 2. Download data and make splits 

In [4]:
if RUN_CONFIG["create_splits"]:
    split_artifacts = create_fixed_split_indices(
        data_root=RUN_CONFIG["data_root"],
        artifacts_root=RUN_CONFIG["artifacts_root"],
        split_seed=int(RUN_CONFIG["split_seed"]),
        val_ratio=float(RUN_CONFIG["val_ratio"]),
        overwrite=bool(RUN_CONFIG["overwrite_splits"]),
        download=bool(RUN_CONFIG["download_data"]),
    )
else:
    split_artifacts = load_fixed_split_indices(artifacts_root=RUN_CONFIG["artifacts_root"])

print("Split artifacts:")
print(f"- train indices: {split_artifacts.train_indices_path}")
print(f"- val indices:   {split_artifacts.val_indices_path}")
print(f"- metadata:      {split_artifacts.metadata_path}")

split_metadata = split_artifacts.metadata
display(pd.DataFrame([split_metadata]))

class_counts = split_metadata.get("class_counts", {})
if isinstance(class_counts, dict) and class_counts:
    class_counts_df = pd.DataFrame(class_counts).fillna(0).astype(int)
    display(class_counts_df)

datasets = build_downstream_datasets(
    train_indices=split_artifacts.train_indices,
    val_indices=split_artifacts.val_indices,
    data_root=RUN_CONFIG["data_root"],
    download=bool(RUN_CONFIG["download_data"]),
)

batch = next(iter(DataLoader(datasets.train, batch_size=8, shuffle=False, num_workers=0)))
images, labels = batch
print(f"Train sample batch image shape: {tuple(images.shape)}")
print(f"Train sample batch label range: [{int(labels.min())}, {int(labels.max())}]")
print(f"Train subset size: {len(datasets.train)}")
print(f"Val subset size:   {len(datasets.val)}")
print(f"Test set size:     {len(datasets.test)}")

Split artifacts:
- train indices: /Users/tlam/cv/artifacts/splits/stl10_train_indices.json
- val indices:   /Users/tlam/cv/artifacts/splits/stl10_val_indices.json
- metadata:      /Users/tlam/cv/artifacts/splits/stl10_split_metadata.json


,dataset,source_split,num_classes,split_seed,stratified,train_ratio,val_ratio,train_count,val_count,class_counts,note
0,STL10,train,10,42,True,0.8,0.2,4000,1000,"{'train': {'0': 400, '1': 400, '2': 400, '3': ...",Fixed 80/20 stratified split sampled once with...


,train,val
0,400,100
1,400,100
2,400,100
3,400,100
4,400,100
5,400,100
6,400,100
7,400,100
8,400,100
9,400,100


Train sample batch image shape: (8, 3, 224, 224)
Train sample batch label range: [1, 9]
Train subset size: 4000
Val subset size:   1000
Test set size:     8000


## 4. Single Batch Test for Training and Run Training  

In [5]:
def _recipe_id_for_condition(condition: str) -> str | None:
    if condition == "random_init":
        return RUN_CONFIG["random_init_recipe_id"]
    return RUN_CONFIG["probe_recipe_id"]


def run_cross_condition_one_batch_check(*, conditions: list[str]) -> None:
    device = torch.device(str(RUN_CONFIG["device"]))
    batch_loader = DataLoader(datasets.train, batch_size=8, shuffle=False, num_workers=0)
    images, targets = next(iter(batch_loader))
    images = images.to(device)

    labels_min = int(targets.min().item())
    labels_max = int(targets.max().item())
    if labels_min < 0 or labels_max > 9:
        raise RuntimeError(
            f"Unexpected label range: min={labels_min}, max={labels_max}"
        )

    for condition in conditions:
        recipe = resolve_training_recipe(
            condition=condition,
            recipe_id=_recipe_id_for_condition(condition),
        )
        mode_config = resolve_mode_config(
            condition=condition,
            training_mode=recipe.training_mode,
        )

        checkpoint_path = None
        if condition == "moco":
            checkpoint_path = str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path)
        elif condition == "swav":
            checkpoint_path = str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path)

        model = build_downstream_model(
            condition=condition,
            num_classes=10,
            freeze_encoder=mode_config.freeze_encoder,
            trainable_layer4=mode_config.trainable_layer4,
            device=device,
            allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
            checkpoint_path=checkpoint_path,
        )
        model.eval()
        with torch.no_grad():
            logits = model(images)

        expected_shape = (images.shape[0], 10)
        if tuple(logits.shape) != expected_shape:
            raise RuntimeError(
                f"Unexpected logits shape for {condition}: {tuple(logits.shape)}"
            )


if RUN_CONFIG["run_cross_condition_sanity_check"]:
    run_cross_condition_one_batch_check(conditions=list(RUN_CONFIG["conditions"]))
    print("Cross-condition one-batch sanity check passed.")
else:
    print("Skipping cross-condition one-batch sanity check.")

run_results: list[dict[str, object]] = []
if RUN_CONFIG["run_training"]:
    for condition in list(RUN_CONFIG["conditions"]):
        recipe_id = _recipe_id_for_condition(condition)
        for seed in list(RUN_CONFIG["seeds"]):
            config = TrainingRunConfig(
                condition=condition,
                seed=int(seed),
                recipe_id=recipe_id,
                device=str(RUN_CONFIG["device"]),
                artifacts_root=RUN_CONFIG["artifacts_root"],
                data_root=RUN_CONFIG["data_root"],
                download=bool(RUN_CONFIG["download_data"]),
                num_workers=int(RUN_CONFIG["num_workers"]),
                pin_memory=bool(RUN_CONFIG["pin_memory"]),
                allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
                moco_checkpoint_path=str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path),
                swav_checkpoint_path=str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path),
                sanity_checks=bool(RUN_CONFIG["sanity_checks"]),
            )
            print(f"[run] condition={condition} seed={seed}")
            result = train_one_run(config)
            run_results.append(result)
            print(
                f"[done] condition={condition} seed={seed} "
                f"best_val_acc={result['best_val_acc']:.4f} test_acc={result['test_acc']:.4f}"
            )
else:
    print("Skipping training (run_training=False).")

Cross-condition one-batch sanity check passed.
[run] condition=supervised seed=0


KeyboardInterrupt: 

### Loss curves for each training 

In [ ]:
def _resolve_loss_paths_from_run(run: dict[str, object]) -> tuple[Path | None, Path | None]:
    batch_csv = run.get("batch_loss_csv_path")
    epoch_csv = run.get("epoch_loss_csv_path")
    metrics_path = run.get("metrics_path")

    batch_path = Path(batch_csv) if isinstance(batch_csv, str) and batch_csv else None
    epoch_path = Path(epoch_csv) if isinstance(epoch_csv, str) and epoch_csv else None

    if (batch_path is None or epoch_path is None) and isinstance(metrics_path, str):
        metrics = Path(metrics_path)
        stem = metrics.stem
        base = metrics.parent
        if batch_path is None:
            candidate = base / f"{stem}.batch_losses.csv"
            if candidate.exists():
                batch_path = candidate
        if epoch_path is None:
            candidate = base / f"{stem}.epoch_losses.csv"
            if candidate.exists():
                epoch_path = candidate

    return batch_path, epoch_path


loss_source_runs = list(run_results)
if not loss_source_runs:
    print("No results found")
    fallback_runs: list[dict[str, object]] = []
    run_metrics_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "probe_runs"
    if run_metrics_root.exists():
        for condition_dir in sorted(run_metrics_root.glob("*")):
            if not condition_dir.is_dir():
                continue
            condition = condition_dir.name
            if condition not in set(list(RUN_CONFIG["conditions"])):
                continue
            for json_path in sorted(condition_dir.glob("*.json")):
                payload = read_json(json_path)
                if not isinstance(payload, dict):
                    continue
                seed_value = payload.get("seed")
                if not isinstance(seed_value, int):
                    continue
                if seed_value not in [int(seed) for seed in list(RUN_CONFIG["seeds"])]:
                    continue
                payload["metrics_path"] = str(json_path)
                fallback_runs.append(payload)
    loss_source_runs = fallback_runs

if not loss_source_runs:
    print("No training losses found")
else:
    for run in loss_source_runs:
        condition = str(run.get("condition", "unknown"))
        seed = int(run.get("seed", -1))
        recipe_id = str(run.get("recipe_id", "unknown_recipe"))

        batch_path, epoch_path = _resolve_loss_paths_from_run(run)
        print(f"[loss-curves] condition={condition} seed={seed} recipe={recipe_id}")
        print(f"- batch CSV: {batch_path}")
        print(f"- epoch CSV: {epoch_path}")

        if batch_path is None or epoch_path is None:
            print("  [skip] Missing loss paths in run payload.")
            continue
        if not batch_path.exists() or not epoch_path.exists():
            print("  [skip] Loss CSV files do not exist on disk.")
            continue

        batch_df = pd.read_csv(batch_path)
        epoch_df = pd.read_csv(epoch_path)
        if batch_df.empty or epoch_df.empty:
            print("  [skip] One of the loss CSV files is empty.")
            continue

        fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

        axes[0].plot(
            batch_df["global_step"],
            batch_df["train_loss"],
            alpha=0.30,
            linewidth=1.0,
            label="batch loss",
        )
        smooth = batch_df["train_loss"].rolling(window=20, min_periods=1).mean()
        axes[0].plot(
            batch_df["global_step"],
            smooth,
            linewidth=2.0,
            label="batch loss (rolling-20)",
        )
        axes[0].set_title(f"{condition} seed={seed} - Batch Loss")
        axes[0].set_xlabel("Global Step")
        axes[0].set_ylabel("Loss")
        axes[0].grid(alpha=0.25)
        axes[0].legend()

        axes[1].plot(
            epoch_df["epoch"],
            epoch_df["train_loss"],
            marker="o",
            linewidth=2.0,
            label="train loss",
        )
        if "val_loss" in epoch_df.columns and epoch_df["val_loss"].notna().any():
            axes[1].plot(
                epoch_df["epoch"],
                epoch_df["val_loss"],
                marker="o",
                linewidth=2.0,
                label="val loss",
            )
        axes[1].set_title(f"{condition} seed={seed} - Epoch Loss")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].grid(alpha=0.25)
        axes[1].legend()

        plt.show()

## Load Existing Run Metrics, Summarize, and Plot

In [ ]:
def load_existing_run_results(
    *,
    run_metrics_root: Path,
    conditions: list[str],
    seeds: list[int],
) -> list[dict[str, object]]:
    if not run_metrics_root.exists():
        return []

    allowed_conditions = set(conditions)
    allowed_seeds = set(seeds)
    payloads: list[dict[str, object]] = []

    for condition_dir in sorted(run_metrics_root.glob("*")):
        if not condition_dir.is_dir():
            continue
        condition = condition_dir.name
        if condition not in allowed_conditions:
            continue

        for json_path in sorted(condition_dir.glob("*.json")):
            payload = read_json(json_path)
            if not isinstance(payload, dict):
                continue
            seed = payload.get("seed")
            if not isinstance(seed, int):
                continue
            if seed not in allowed_seeds:
                continue
            payloads.append(payload)

    return payloads


run_metrics_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "probe_runs"
existing_results = load_existing_run_results(
    run_metrics_root=run_metrics_root,
    conditions=list(RUN_CONFIG["conditions"]),
    seeds=[int(seed) for seed in list(RUN_CONFIG["seeds"])],
)

# Prefer fresh in-memory results when a condition+seed key overlaps.
combined_by_key: dict[tuple[str, int], dict[str, object]] = {}
for payload in existing_results:
    key = (str(payload["condition"]), int(payload["seed"]))
    combined_by_key[key] = payload
for payload in run_results:
    key = (str(payload["condition"]), int(payload["seed"]))
    combined_by_key[key] = payload

combined_results = [combined_by_key[key] for key in sorted(combined_by_key)]
if not combined_results:
    print("No run results found. Run training or point to existing artifacts first.")
else:
    run_rows = [build_run_table_row(payload) for payload in combined_results]
    run_df = pd.DataFrame(run_rows).sort_values(["condition", "seed"]).reset_index(drop=True)
    display(run_df)

    summaries = summarize_test_accuracy_by_condition(run_rows)
    summary_df = pd.DataFrame(summaries).sort_values(["condition"]).reset_index(drop=True)
    display(summary_df)

    output_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "probe_runs"
    ensure_parent(output_root / "placeholder.txt")

    run_table_json = output_root / "run_table.notebook.json"
    run_table_csv = output_root / "run_table.notebook.csv"
    summary_json = output_root / "condition_summary.notebook.json"
    summary_csv = output_root / "condition_summary.notebook.csv"

    write_json(run_table_json, run_rows)
    run_df.to_csv(run_table_csv, index=False)
    write_json(summary_json, summaries)
    summary_df.to_csv(summary_csv, index=False)

    print(f"Saved run table JSON: {run_table_json}")
    print(f"Saved run table CSV:  {run_table_csv}")
    print(f"Saved summary JSON:   {summary_json}")
    print(f"Saved summary CSV:    {summary_csv}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(summary_df["condition"], summary_df["mean_test_acc"], yerr=summary_df["std_test_acc"], capsize=4)
    ax.set_title("Mean +/- Std Test Accuracy by Condition")
    ax.set_xlabel("Condition")
    ax.set_ylabel("Test Top-1 Accuracy")
    ax.set_ylim(0.0, 1.0)
    ax.grid(axis="y", alpha=0.25)
    plt.show()

## Stages 5-6 - Generate Explanations and Run QC

In [ ]:
EXPLAIN_CONFIG: dict[str, object] = {
    "run_explanations": False,
    "run_qc": False,
    "conditions": list(RUN_CONFIG["conditions"]),
    "seeds": [int(seed) for seed in list(RUN_CONFIG["seeds"])],
    "methods": ["gradcam", "gradcampp", "occlusion"],
    "batch_size": 8,
    "overwrite": False,
}

print(json.dumps(EXPLAIN_CONFIG, indent=2))

if EXPLAIN_CONFIG["run_explanations"]:
    runs = discover_stage4_runs(
        artifacts_root=RUN_CONFIG["artifacts_root"],
        conditions=list(EXPLAIN_CONFIG["conditions"]),
        seeds=list(EXPLAIN_CONFIG["seeds"]),
    )
    saliency_rows = generate_explanations_for_runs(
        runs=runs,
        methods=list(EXPLAIN_CONFIG["methods"]),
        batch_size=int(EXPLAIN_CONFIG["batch_size"]),
        data_root=RUN_CONFIG["data_root"],
        artifacts_root=RUN_CONFIG["artifacts_root"],
        device=str(RUN_CONFIG["device"]),
        allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
        overwrite=bool(EXPLAIN_CONFIG["overwrite"]),
        download=bool(RUN_CONFIG["download_data"]),
    )
    print(f"Generated saliency rows: {len(saliency_rows)}")
else:
    print("Skipping explanation generation (run_explanations=False).")

if EXPLAIN_CONFIG["run_qc"]:
    qc_report = run_explanation_qc(
        artifacts_root=RUN_CONFIG["artifacts_root"],
        conditions=list(EXPLAIN_CONFIG["conditions"]),
        seeds=list(EXPLAIN_CONFIG["seeds"]),
        methods=list(EXPLAIN_CONFIG["methods"]),
    )
    qc_report_path = write_explanation_qc_report(
        report=qc_report,
        artifacts_root=RUN_CONFIG["artifacts_root"],
    )
    print(f"Saved QC report: {qc_report_path}")
    display(pd.DataFrame(qc_report.get("coverage", [])))
    if qc_report.get("errors"):
        display(pd.DataFrame(qc_report["errors"]))
else:
    print("Skipping explanation QC (run_qc=False).")